In [0]:
from pyspark.sql import functions as F

# Load tables
fact_encounters = spark.table("medical_data.gold.fact_encounters")
dim_procedure = spark.table("medical_data.gold.dim_procedure")
dim_encounter = spark.table("medical_data.gold.dim_encounter")

# encounter_base — derive date fields from start, join dim_encounter for encounter_class
encounter_base = (
    fact_encounters
    .join(dim_encounter, "encounter_id", "left")
    .select(
        F.year("start").alias("year"),
        fact_encounters["quarter"],
        fact_encounters["month"],
        F.when(F.month("start") <= 6, 1).otherwise(2).alias("half_year"),
        fact_encounters["patient_id"],
        fact_encounters["payer_id"],
        fact_encounters["encounter_id"],
        dim_encounter["encounter_class"],
        fact_encounters["total_claim_cost"],
        fact_encounters["base_encounter_cost"],
        (F.col("payer_coverage") > 0).cast("boolean").alias("has_payer_coverage"),
        F.datediff(fact_encounters["stop"], fact_encounters["start"]).alias("encounter_duration_days")
    )
)

# procedure_agg — count procedures per encounter using dim_procedure
procedure_agg = (
    dim_procedure
    .groupBy("encounter_id")
    .agg(
        F.count("*").alias("procedure_count")
    )
)

# Final aggregation
datacube = (
    encounter_base
    .join(
        procedure_agg,
        encounter_base["encounter_id"] == procedure_agg["encounter_id"],
        "left"
    )
    .groupBy(
        encounter_base["year"], 
        encounter_base["quarter"], 
        encounter_base["month"], 
        encounter_base["half_year"], 
        encounter_base["payer_id"], 
        encounter_base["encounter_class"]
    )
    .agg(
        F.countDistinct(encounter_base["encounter_id"]).alias("total_encounters"),
        F.countDistinct(encounter_base["patient_id"]).alias("unique_patients"),
        F.sum(encounter_base["total_claim_cost"]).alias("total_encounter_cost"),
        F.sum(F.coalesce(procedure_agg["procedure_count"], F.lit(0))).alias("total_procedures"),
        F.sum(encounter_base["base_encounter_cost"]).alias("total_procedure_cost")
    )
    .orderBy(encounter_base["year"], encounter_base["quarter"], encounter_base["month"], encounter_base["encounter_class"])
)

display(datacube)

# Save table
datacube.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("medical_data.gold.data_cube")

In [0]:
%sql
select * from medical_data.gold.data_cube ;